# Regression metrics: MAE, MSE, RMSE, R², adjusted R²

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("data/placement.csv")
X_train, X_test, y_train, y_test = train_test_split(df[["cgpa"]], df["package"], test_size=0.2, random_state=2)
lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)

## The four metrics with scikit-learn

In [ ]:
print("MAE ", mean_absolute_error(y_test, y_pred))
print("MSE ", mean_squared_error(y_test, y_pred))
print("RMSE", root_mean_squared_error(y_test, y_pred))
print("R2  ", r2_score(y_test, y_pred))

## The same by hand

In [ ]:
err = y_test.to_numpy() - y_pred
mae = np.abs(err).mean()
mse = (err ** 2).mean()
rmse = np.sqrt(mse)
ss_res = (err ** 2).sum()                                   # squared errors of the line
ss_tot = ((y_test - y_test.mean()) ** 2).sum()              # squared errors of 'guess the mean'
r2 = 1 - ss_res / ss_tot
print(mae, mse, rmse, ss_res, ss_tot, r2)

## Adjusted R²

In [ ]:
def adjusted_r2(r2, n, k):
    # n rows, k input columns
    return 1 - (1 - r2) * (n - 1) / (n - 1 - k)

adjusted_r2(r2, n=len(y_test), k=1)

## Adding useless random columns (fixed seed)

In [ ]:
noise = np.random.default_rng(0).random((len(df), 20))
X_all = np.c_[df.cgpa.to_numpy(), noise]
Xtr, Xte, ytr, yte = train_test_split(X_all, df.package.to_numpy(), test_size=0.2, random_state=2)
rows = []
for k in [0, 1, 5, 10, 20]:
    m = LinearRegression().fit(Xtr[:, :1 + k], ytr)
    r_tr = r2_score(ytr, m.predict(Xtr[:, :1 + k]))
    r_te = r2_score(yte, m.predict(Xte[:, :1 + k]))
    rows.append({"random cols": k, "R2 train": r_tr, "adj R2 train": adjusted_r2(r_tr, len(ytr), 1 + k),
                 "R2 test": r_te, "adj R2 test": adjusted_r2(r_te, len(yte), 1 + k)})
pd.DataFrame(rows).round(3)